In [1]:
# Memory-Enhanced Multi-Agent Graph using LangGraph
# Features: Global Memory, Per-Agent Memory, Shared Context
import os
from dotenv import load_dotenv
from typing import TypedDict

from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain.memory import ConversationBufferMemory
from langchain_core.runnables import RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load Gemini API Key
load_dotenv()

True

In [3]:
# Global LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0.9
)

In [4]:
# Shared State
class MemoryState(TypedDict):
    input: str
    planner_memory: str
    researcher_memory: str
    writer_memory: str
    output: str

In [5]:
# Per-Agent Memory Buffers
planner_memory = ConversationBufferMemory(memory_key="planner_memory")
researcher_memory = ConversationBufferMemory(memory_key="researcher_memory")
writer_memory = ConversationBufferMemory(memory_key="writer_memory")

C:\Users\emon1\AppData\Local\Temp\ipykernel_9196\900678697.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  planner_memory = ConversationBufferMemory(memory_key="planner_memory")


In [6]:
# Planner Agent
def planner_node(state: MemoryState) -> MemoryState:
    query = state["input"]
    conversation = planner_memory.load_memory_variables({})["planner_memory"]
    prompt = f"""
    You are a planning agent. You remember previous plans:
    {conversation}

    Plan the following question in steps:
    Question: {query}
    """
    response = llm.invoke(prompt).content.strip()
    planner_memory.save_context({"input": query}, {"planner_memory": response})
    state["planner_memory"] = response
    return state

In [7]:
# Researcher Agent
def researcher_node(state: MemoryState) -> MemoryState:
    steps = state["planner_memory"]
    conversation = researcher_memory.load_memory_variables({})["researcher_memory"]
    prompt = f"""
    You are a research agent. Past research notes:
    {conversation}

    Now research the following steps:
    {steps}
    """
    response = llm.invoke(prompt).content.strip()
    researcher_memory.save_context({"input": steps}, {"researcher_memory": response})
    state["researcher_memory"] = response
    return state

In [8]:
# Writer Agent
def writer_node(state: MemoryState) -> MemoryState:
    research = state["researcher_memory"]
    previous_notes = writer_memory.load_memory_variables({})["writer_memory"]
    prompt = f"""
    You are a writing agent. Previously written:
    {previous_notes}

    Write a final summary using this research:
    {research}
    """
    response = llm.invoke(prompt).content.strip()
    writer_memory.save_context({"input": research}, {"writer_memory": response})
    state["writer_memory"] = response
    state["output"] = response
    return state

In [9]:
# LangGraph Setup
builder = StateGraph(MemoryState)
builder.add_node("planner", planner_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)

In [10]:
builder.set_entry_point("planner")
builder.add_edge("planner", "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)

In [11]:
graph = builder.compile()

In [12]:
# Run Example
query = {
    "input": "What are the risks and benefits of AI in warfare?",
    "planner_memory": "",
    "researcher_memory": "",
    "writer_memory": "",
    "output": ""
}

In [13]:
final_result = graph.invoke(query)
print("\nFINAL OUTPUT:\n", final_result["output"])


FINAL OUTPUT:
 Of course. Here is the final summary based on the provided research.

***

### **Final Summary: The Dual-Edged Sword of AI in Warfare**

Artificial intelligence is fundamentally reshaping the character of modern warfare, presenting a stark dual-use dilemma. Its integration into military operations offers significant potential benefits but is shadowed by profound and potentially catastrophic risks.

On one hand, AI promises to make warfare more precise, efficient, and safer for human soldiers. AI-powered systems can enhance targeting accuracy to reduce civilian casualties, while autonomous drones and vehicles can perform dangerous reconnaissance or disposal tasks, keeping personnel out of harm's way. By processing vast amounts of intelligence data at machine speed, AI can grant commanders a decisive "decision advantage," while also optimizing complex logistics and bolstering cyber defenses.

However, these advantages are counterbalanced by grave ethical, strategic, and t